# Spam Detection Filter using Machine Learning

**The story in one line:** we teach a model to read a message and decide *spam or genuine* — then we interrogate it honestly: baselines, cross-domain tests, disguised spam, and messages it has never seen anything like.

Run `python train.py` for the full pipeline (tuning, threshold selection, cross-domain, robustness). This notebook walks through the same ideas interactively with one fast model.

## 1. The data

The **UCI SMS Spam Collection**: 5,572 real text messages labelled spam/ham. First job: remove **exact duplicates**. Why? If the same message sits in both train and test, the model can 'memorise' test answers and every score becomes a lie. After dedupe: 5,171 messages, ~12.6% spam.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.data import load_sms, load_emails, load_indian_test_set

sms = load_sms()
print(sms['label'].value_counts().to_dict())
sms.head(3)

## 2. Cleaning — tokens, not deletion

A beginner's instinct is to delete numbers and links. But in spam, the *shape* is the signal: a message full of links and phone numbers is suspicious even before you read the words. So we **replace** them with tokens — `urltoken`, `phonetoken`, `numtoken` — and lowercase everything.

In [ ]:
from src.features import clean_text
samples = [
    'WINNER!! Claim your FREE cash at www.prize-now.com or call 09061701461 NOW!',
    'Your OTP is 482913, valid for 10 minutes. Do not share.',
]
for s in samples:
    print('BEFORE:', s)
    print('AFTER :', clean_text(s), '\n')

## 3. Split first, touch test last

Stratified split: **70% train / 15% validation / 15% test**. Stratified = every split keeps the same spam ratio. The test set stays untouched until the final evaluation — it is our exam paper. The validation set exists for one job: choosing the decision threshold.

In [ ]:
from sklearn.model_selection import train_test_split
X = sms[['text']]; y = sms['label'].to_numpy()
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15/0.85, stratify=y_tv, random_state=42)
print(len(X_train), len(X_val), len(X_test))

## 4. Features — three views of one message

1. **Word TF-IDF (1–2 grams)** — which words/phrases appear.
2. **Character TF-IDF (3–5 grams)** — catches disguises like `fr33` or `c@$h` that word features have never seen.
3. **Handcrafted numbers** — length, uppercase ratio, digit ratio, URL count, currency symbols (spam has a 'shape').

Everything lives inside one sklearn **Pipeline**, so vectorizers and scalers are fitted on training data only — no leakage.

## 5. Baselines before models

Two dumb baselines set the bar: (a) always predict ham — it scores ~87% accuracy while catching **zero** spam, which is why accuracy alone is a trap on imbalanced data; (b) keyword rules — spam if ≥2 spammy keywords appear.

In [ ]:
from src.models import majority_predict, keyword_predict
from src.evaluate import metric_dict
maj = metric_dict(y_test, majority_predict(y_train, len(y_test)))
kw_pred, kw_score = keyword_predict(X_test['text'])
kw = metric_dict(y_test, kw_pred, kw_score)
print('Majority baseline  F1:', maj['f1'], '| accuracy:', maj['accuracy'])
print('Keyword baseline   F1:', kw['f1'], '| accuracy:', kw['accuracy'])

## 6. A real model (fast demo: Naive Bayes)

`train.py` tunes five models (Naive Bayes, Logistic Regression, Linear SVM, Random Forest, XGBoost) with cross-validation optimising **PR-AUC**. Here we fit just Naive Bayes to see the jump over baselines.

In [ ]:
from src.models import get_model_specs
pipe, grid = get_model_specs()['Naive Bayes']
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]
print(metric_dict(y_test, (proba >= 0.5).astype(int), proba))

## 7. The threshold is a decision, not a default

0.5 is arbitrary. A **false positive** (your real message lands in the spam folder) hurts more than a missed spam, so on the *validation* set we choose the threshold with the best F1 among those keeping **precision ≥ 0.95**. In the full run this picked **0.831** for the winning XGBoost model.

In [ ]:
from src.evaluate import choose_threshold
val_proba = pipe.predict_proba(X_val)[:, 1]
print(choose_threshold(y_val, val_proba, min_precision=0.95))

## 8. Cross-domain: SMS brain, email world

Train on SMS, test on emails (SpamAssassin corpus) — and the reverse. Scores fall hard (F1 ~0.95 → ~0.63 and ~0.38 in the full run): different era, different vocabulary, different tricks. **Lesson:** a model knows only the world it trained on; production filters retrain on fresh data constantly.

In [ ]:
try:
    email = load_emails()
    p = pipe.predict_proba(email[['text']])[:, 1]
    print('SMS-trained Naive Bayes on ALL emails:', metric_dict(email['label'].to_numpy(), (p >= 0.5).astype(int), p))
except Exception as e:
    print('Email corpus unavailable:', e)

## 9. Disguised spam & modern Indian scams

Attackers disguise words (`fr33 m0ney`, `F R E E`, `f.r.e.e`) and invent new scripts (KYC updates, courier fees, UPI cashback). Our hand-written set of 30 modern Indian messages is the hardest test in the project — the full model gets about 2/3 right. Honest weakness, clear cause: 2020s scam vocabulary barely exists in 2000s training data.

In [ ]:
indian = load_indian_test_set()
try:
    import joblib
    final = joblib.load('../models/spam_pipeline.joblib')
    meta = __import__('json').load(open('../models/metadata.json'))
    pr = final.predict_proba(indian[['text']])[:, 1]
    print(metric_dict(indian['label'].to_numpy(), (pr >= meta['threshold']).astype(int), pr))
except Exception as e:
    print('Run train.py first:', e)

## 10. What did the model learn?

Word evidence (spam vs ham log-odds) plus LIME explanations for single messages live in `reports/figures/` (`top_words.png`, `lime_example.png`) and in the app's **Model Insights** tab.

## Where to go next
- `train.py` — the complete, reproducible pipeline
- `app/app.py` — the live demo (`streamlit run app/app.py`)
- `README.md` — results table and every design decision